<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_00_environment_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 00 — Environment Check

**Deep Learning for Engineering · Aalborg University · Part 1**

Run this notebook first, top to bottom. There is nothing to write in it: every
cell is complete. Its job is to establish, before you spend an hour on anything
else, that

* `torch`, `numpy` and `matplotlib` are importable and recent enough;
* `Ex_5_core.py` is on the path and imports cleanly;
* the four datasets this exercise set uses can be generated on **your** machine,
  with no network connection;
* a convolution, a graph and a recurrent step all run.

**Nothing is downloaded.** That is a deliberate choice rather than an oversight.
The usual first exercise in a convolutional-network course fetches MNIST from a
mirror, and every year that mirror is slow, blocked by a university firewall, or
has moved. The three image classes here are drawn procedurally in about twenty
lines of NumPy, and they are the same three classes on every machine in the
room.

If a cell in this notebook fails, fix it before going on. A missing package is a
two-minute problem now and a lost afternoon in notebook 03.

---

## 0 · Versions

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("torch      ", torch.__version__)
print("cuda        available:", torch.cuda.is_available(), " (not needed)")

**What you should see.** Four version numbers and `cuda available: False`
on most machines.

Any Python from 3.9 and any PyTorch from 2.0 will do. **No GPU is required
anywhere in Ex_05.** The largest model in this exercise set has about two
thousand parameters; a GPU would spend more time moving the data than computing
on it.

---

## 1 · The shared module

In [ ]:
import Ex_5_core as core

print("module file:", core.__file__)
print("output dir :", core.OUTPUT_DIR)
print()
print("classes    :", core.CLASS_NAMES)
print("buses      :", core.BUS_NAMES)
print("lines      :", len(core.SIX_BUS_LINES))

**What you should see.** The path to `Ex_5_core.py`, a path ending in
`Ex05_outputs`, the three image classes `('clean', 'crack', 'pit')`, the six bus
names, and `lines : 8`.

If the import fails with `ModuleNotFoundError`, the notebook is not being run
from the `Ex05-cnn-and-gnn` folder. On Colab, upload the whole folder and
`%cd` into it.

---

## 2 · Dataset one — the weld radiographs

Sixty images, twenty of each class, 16 by 16 pixels, generated here and now.

In [ ]:
X_img, y_img = core.weld_images(n_per_class=20, seed=3)

print("images :", X_img.shape, X_img.dtype)
print("labels :", y_img.shape, "counts", np.bincount(y_img))
print("pixels : min %.3f  max %.3f" % (X_img.min(), X_img.max()))

core.plot_images(X_img, y_img, n=12, title="Twelve synthetic weld radiographs")
plt.show()

**What you should see.** `images : (60, 1, 16, 16) float32`, label counts
`[20 20 20]`, pixel values inside [0, 1], and a grid of twelve small grey
squares: some plain, some crossed by a thin dark line, some with a small dark
dot.

The shape `(60, 1, 16, 16)` is `(batch, channels, height, width)`. The `1` is
the single greyscale channel. `torch.nn.Conv2d` insists on that four-dimensional
layout, and forgetting the channel axis is the most common first error with a
convolutional network — you will meet it in notebook 01 whether you want to or
not.

Look at the pictures for a moment. The defect is in a different place in every
image. That is the whole reason a convolution is the right tool: the same
detector has to work everywhere on the plate.

---

## 3 · Dataset two — the six-bus network

The object notebook 03 is built around, and the same one **Ex_12.1 in Part 2**
uses.

In [ ]:
A = core.six_bus_adjacency()
B = core.susceptance_matrix()

print("adjacency (6 x 6):")
print(A.astype(int))
print()
print("degrees:", A.sum(axis=1).astype(int))
print("symmetric:", np.allclose(A, A.T))
print()
print("susceptance matrix B, row sums:", np.round(B.sum(axis=1), 12))

core.plot_graph(title="The six-bus network")
plt.show()

**What you should see.** A symmetric matrix of zeros and ones with a zero
diagonal, degrees `[2 3 3 3 3 2]`, `symmetric: True`, susceptance row sums that
are all zero, and a picture of six circles joined by eight lines.

Two of those facts are physics rather than bookkeeping.

**Symmetry.** A transmission line carries power in both directions, so the graph
is undirected. Not every engineering graph is: a pipe network with check valves
is not, and a road network with one-way streets is not.

**Zero row sums.** $B$ is a weighted graph Laplacian, and a Laplacian applied to
the vector of all ones gives zero. Physically: if every bus angle shifts by the
same amount, nothing flows differently, because only *differences* of angle
drive power. That is why $B$ is singular, and why the load flow in notebook 03
needs a reference bus.

---

## 4 · Dataset three — the load profile

In [ ]:
series = core.load_profile(n_days=10, seed=21)

print("series :", series.shape, "  min %.3f  max %.3f  mean %.3f"
      % (series.min(), series.max(), series.mean()))

X_seq, y_seq = core.make_windows(series, window=24)
print("windows:", X_seq.shape, "targets:", y_seq.shape)

core.plot_series(series, title="Ten days of synthetic substation demand",
                 highlight=(48, 72))
plt.show()

**What you should see.** `series : (240,)` with values roughly between 0.2
and 0.85, `windows: (216, 24, 1)` and `targets: (216, 1)`, and a wiggly trace
with a clear daily rhythm — a morning shoulder and a taller evening peak — and
two lighter days at the weekend.

`(216, 24, 1)` is `(batch, time, features)`, which is what every recurrent layer
in PyTorch expects when you pass `batch_first=True`. One feature, because this
is a single measured channel.

---

## 5 · One convolution, one graph step, one recurrent step

Three lines of arithmetic, one from each half of lecture block L5. If these
three cells run, everything in Ex_05 will run.

In [ ]:
import torch.nn as nn

core.set_seed(0)

# (a) a convolution: one 3x3 kernel over one 16x16 image
conv = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1)
out_conv = conv(torch.tensor(X_img[:2]))
print("(a) conv    ", tuple(torch.tensor(X_img[:2]).shape), "->",
      tuple(out_conv.shape), " parameters:", core.count_parameters(conv))

# (b) a message-passing step: multiply node features by the normalised adjacency
A_hat = core.normalised_adjacency(A)
H = torch.tensor(np.eye(6, dtype=np.float32))
out_graph = torch.tensor(A_hat.astype(np.float32)) @ H
print("(b) graph   ", tuple(H.shape), "->", tuple(out_graph.shape),
      " row sums of A_hat:", np.round(A_hat.sum(axis=1), 3))

# (c) a recurrent step: fold one time step into a hidden state
cell = nn.Linear(1 + 8, 8)
h = torch.zeros(5, 8)
x_t = torch.tensor(X_seq[:5, 0, :])
h = torch.tanh(cell(torch.cat([x_t, h], dim=1)))
print("(c) recurrent hidden state ->", tuple(h.shape),
      " parameters:", core.count_parameters(cell))

**What you should see.**

```
(a) conv     (2, 1, 16, 16) -> (2, 4, 16, 16)  parameters: 40
(b) graph    (6, 6) -> (6, 6)  row sums of A_hat: [0.911 1.039 ... ]
(c) recurrent hidden state -> (5, 8)  parameters: 80
```

Three things worth noticing before you close this notebook.

**Forty parameters.** The convolution turned a 16 by 16 image into four 16 by 16
feature maps — 1024 numbers out of 256 — using forty parameters: four kernels of
nine weights, plus four biases. A dense layer doing the same thing would need
$256 \times 1024 = 262{,}144$ weights. That ratio is the argument of L5.1's
first half, and notebook 01 makes you compute it yourself.

**The row sums of $\hat{A}$ are not all one.** They are close to one, and they
differ because the buses have different degrees. This is the symmetric
normalisation $\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ from L5.1; notebook 02
builds it by hand and explains what it is for.

**The recurrent cell has no time index in it.** It is one `Linear` layer, applied
once per time step, with its own previous output fed back in. That single reused
layer *is* the recurrent network — the "unrolling" you saw in L5.2 is a picture
of a `for` loop, not of extra parameters.

---

## 6 · Ready

If every cell above ran, you are set up. Continue with
**`Ex05_01_cnn_image_classification.ipynb`**.

A note on time. Notebook 01 trains a small convolutional network and takes a
minute or two on a laptop CPU. Notebook 03 trains several graph networks and
takes a few minutes. Notebook 04's recurrent models are the slowest thing in the
set, because a `for` loop over twenty-four time steps cannot be vectorised away
— which is itself one of the lecture's points about why attention replaced
recurrence.